In [184]:
import geopandas as gpd
import pandas as pd

import sys
from pathlib import Path

# make sure the notebook can see tree_clustering.py
sys.path.append(str(Path(".").resolve()))

from tree_clustering import (
    METRIC_EPSG,
    SNAP_TOLERANCE_METERS,
    ensure_crs,
    ensure_points,
    add_snapped_centroid_id,
    build_centroids_from_crimes,
)

In [185]:
gdf = gpd.read_file('Data/pois_with_station_lists.geojson')
crime_gdf = gpd.read_file('Data/crimes_with_tree_features.geojson')
print("Crime rows:", len(crime_gdf))

Crime rows: 61873


In [186]:
def classify_poi(row):

    if row["amenity"] in ["bar","pub","nightclub","biergarten","casino","stripclub","music_venue","events_venue"] or row["craft"] == "brewery":
        return "nightlife"

    if row["amenity"] in ["restaurant","fast_food","cafe","food_court","ice_cream"]:
        return "food"

    if row["amenity"] in ["bank","library","post_office","courthouse","waste_disposal","townhall","animal_boarding"]:
        return "errand"
    
    if row["amenity"] in ["place_of_worship","school","fire_station","police","bus_station","shelter","atm","social_facility"]:
        return "public_services"
    
    if row["amenity"] in ["parking","bicycle_parking","bench","waste_basket","drinking_water","telephone","toilets","recycling","vending_machine"]:
        return "env"

    if pd.notna(row["shop"]) or row["amenity"] == "marketplace":
        return "retail"

    if pd.notna(row["leisure"]) or row["amenity"] in ["theatre","cinema","community_centre","fountain"]:
        return "recreation"

    if pd.notna(row["tourism"]):
        return "tourism"

    if pd.notna(row["office"]) or row["amenity"] in ["studio","conference_centre"]:
        return "office"

    if pd.notna(row["healthcare"]) or row["amenity"] == "veterinary":
        return "healthcare"

    return "other"


gdf["poi_cluster"] = gdf.apply(classify_poi, axis=1)

In [187]:
gdf['poi_cluster'].value_counts()

poi_cluster
env                1459
other               479
tourism             310
recreation          297
food                253
retail              130
nightlife            62
public_services      58
office               52
errand               40
healthcare           22
Name: count, dtype: int64

In [188]:
gdf_other = gdf[gdf["poi_cluster"] == "other"]
gdf_other = gdf_other.drop(columns=["shop", "tourism", "office", "healthcare", "craft", "public_transport", "leisure"])
gdf_other.head()

,osm_id,name,lat,lon,tags,amenity,stations_in_radius,stations_in_radius_dist_m,geometry,poi_cluster
21,9875353335,NaN,35.232382,-80.837725,"{ ""amenity"": ""parking_entrance"" }",parking_entrance,[9th Street],[387.25],POINT (-80.83773 35.23238),other
22,9875353339,NaN,35.232059,-80.838176,"{ ""amenity"": ""parking_entrance"" }",parking_entrance,[9th Street],[389.77],POINT (-80.83818 35.23206),other
52,11023996516,NaN,35.230696,-80.839559,"{ ""amenity"": ""parking_entrance"" }",parking_entrance,[9th Street],[429.33],POINT (-80.83956 35.2307),other
107,11355293678,NaN,35.226343,-80.838150,"{ ""amenity"": ""parking_entrance"" }",parking_entrance,"[7th Street, CTC/Arena, 9th Street]","[97.59, 330.34, 469.84]",POINT (-80.83815 35.22634),other
115,12363254606,NaN,35.230598,-80.839285,"{ ""amenity"": ""parking_entrance"" }",parking_entrance,[9th Street],[402.44],POINT (-80.83928 35.2306),other


In [189]:
gdf_other['amenity'].value_counts()

amenity
parking_entrance          277
parking_space              18
car_rental                  4
bicycle_repair_station      4
car_wash                    3
public_bookcase             2
vacant                      2
post_box                    1
clock                       1
fuel                        1
Name: count, dtype: int64

In [190]:
gdf = gdf[gdf["poi_cluster"] != "other"].copy()

gdf_env = gdf[gdf["poi_cluster"] == "env"].copy()
gdf_pois = gdf[gdf["poi_cluster"] != "env"].copy()

In [191]:
print("Crime rows:", len(crime_gdf))
print("Unique geometries:", crime_gdf.geometry.nunique())

Crime rows: 61873
Unique geometries: 196


In [192]:
crime_gdf = ensure_crs(crime_gdf, "crime_gdf")
crime_gdf = ensure_points(crime_gdf, "crime_gdf").copy()

print("Crime rows:", len(crime_gdf))
print("Crime CRS:", crime_gdf.crs)
print("Geometry type counts:")
print(crime_gdf.geometry.geom_type.value_counts())

display(crime_gdf.head())

Crime rows: 61873
Crime CRS: EPSG:4326
Geometry type counts:
Point    61873
Name: count, dtype: int64


,INCIDENT_REPORT_ID,LOCATION,ZIP,LATITUDE_PUBLIC,LONGITUDE_PUBLIC,CMPD_PATROL_DIVISION,NPA,LOCATION_TYPE_DESCRIPTION,PLACE_TYPE_DESCRIPTION,PLACE_DETAIL_DESCRIPTION,...,x,y,x_snap,y_snap,centroid_id,nearest_tree_dist_m,trees_within_10m,trees_within_25m,trees_within_50m,geometry
0,20220704-0820-00,300 S BREVARD ST,28202,35.22,-80.84,Central,476,Outdoors,Open Area,Street/Highway,...,514354.061678,3.897786e+06,514350.0,3897775.0,111,3.523966,3,21,51,POINT (-80.84227 35.22301)
1,20220704-0820-00,300 S BREVARD ST,28202,35.22,-80.84,Central,476,Outdoors,Open Area,Street/Highway,...,514354.061678,3.897786e+06,514350.0,3897775.0,111,3.523966,3,21,51,POINT (-80.84227 35.22301)
2,20220704-0820-00,300 S BREVARD ST,28202,35.22,-80.84,Central,476,Outdoors,Open Area,Street/Highway,...,514354.061678,3.897786e+06,514350.0,3897775.0,111,3.523966,3,21,51,POINT (-80.84227 35.22301)
3,20230104-1325-00,400 S TRYON ST,28202,35.22,-80.85,Central,476,Outdoors,Commercial Place,Other - Commercial Place,...,513979.527198,3.897997e+06,513975.0,3898000.0,92,15.400769,0,12,46,POINT (-80.84638 35.22492)
4,20230104-1325-00,400 S TRYON ST,28202,35.22,-80.85,Central,476,Outdoors,Commercial Place,Other - Commercial Place,...,513979.527198,3.897997e+06,513975.0,3898000.0,92,15.400769,0,12,46,POINT (-80.84638 35.22492)


In [193]:
crimes_m = add_snapped_centroid_id(crime_gdf)
crime_centroids = build_centroids_from_crimes(crimes_m)

centroids = (
    crimes_m
    .groupby("centroid_id")[["x_snap", "y_snap"]]
    .first()
    .reset_index()
)

centroids = gpd.GeoDataFrame(
    centroids,
    geometry=gpd.points_from_xy(centroids["x_snap"], centroids["y_snap"]),
    crs=f"EPSG:{METRIC_EPSG}"
)

print("Snapping tolerance (meters):", SNAP_TOLERANCE_METERS)
print("Crime rows:", len(crimes_m))
print("Unique centroid_ids:", crimes_m["centroid_id"].nunique())
print("Crime centroids built:", len(crime_centroids))

display(crime_centroids.head())

Snapping tolerance (meters): 25
Crime rows: 61873
Unique centroid_ids: 184
Crime centroids built: 184


,centroid_id,x_snap,y_snap,geometry
0,111,514350.0,3897775.0,POINT (514350 3897775)
1,92,513975.0,3898000.0,POINT (513975 3898000)
2,181,515325.0,3898675.0,POINT (515325 3898675)
3,81,513775.0,3897825.0,POINT (513775 3897825)
4,54,513250.0,3897125.0,POINT (513250 3897125)


In [194]:
crimes_check = crime_gdf.to_crs(epsg=METRIC_EPSG).copy()
crimes_check["x"] = crimes_check.geometry.x
crimes_check["y"] = crimes_check.geometry.y
crimes_check["x_snap"] = (crimes_check["x"] / SNAP_TOLERANCE_METERS).round() * SNAP_TOLERANCE_METERS
crimes_check["y_snap"] = (crimes_check["y"] / SNAP_TOLERANCE_METERS).round() * SNAP_TOLERANCE_METERS

print("Unique raw x/y:", crimes_check[["x", "y"]].drop_duplicates().shape[0])
print("Unique snapped x/y:", crimes_check[["x_snap", "y_snap"]].drop_duplicates().shape[0])

display(
    crimes_check[["x", "y", "x_snap", "y_snap"]].head(15)
)

ENV_RADII_M = [10, 25, 50]

Unique raw x/y: 196
Unique snapped x/y: 184


,x,y,x_snap,y_snap
0,514354.061678,3.897786e+06,514350.0,3897775.0
1,514354.061678,3.897786e+06,514350.0,3897775.0
2,514354.061678,3.897786e+06,514350.0,3897775.0
3,513979.527198,3.897997e+06,513975.0,3898000.0
4,513979.527198,3.897997e+06,513975.0,3898000.0
5,515323.215726,3.898666e+06,515325.0,3898675.0
6,513785.032030,3.897834e+06,513775.0,3897825.0
7,513261.448442,3.897119e+06,513250.0,3897125.0
8,514773.742530,3.898861e+06,514775.0,3898850.0
9,515201.980857,3.898196e+06,515200.0,3898200.0


In [195]:
def compute_env_features(centroids, gdf_evn, env_radii_m=[10, 25, 50]):
    env_m = gdf_evn.to_crs(epsg=METRIC_EPSG).copy()

    # keep only valid points
    env_m = env_m[~env_m.geometry.isna()].copy()
    env_m = env_m[env_m.geometry.geom_type == "Point"].copy()

    # force one row per centroid_id
    centroids_u = (
        centroids[["centroid_id", "geometry"]]
        .drop_duplicates(subset=["centroid_id"])
        .copy()
    )

    feats = centroids_u.copy()

    for r in env_radii_m:
        buffers = centroids_u.copy()
        buffers["geometry"] = buffers.geometry.buffer(r)

        j = gpd.sjoin(
            env_m[["geometry"]],
            buffers,
            how="inner",
            predicate="within"
        )

        counts = (
            j.groupby("centroid_id")
            .size()
            .rename(f"env_within_{r}m")
            .reset_index()
        )

        feats = feats.merge(counts, on="centroid_id", how="left")

    for r in env_radii_m:
        col = f"env_within_{r}m"
        feats[col] = feats[col].fillna(0).astype(int)

    feats["env_object"] = feats["env_within_50m"]

    return feats.drop(columns="geometry")

In [196]:
env_centroid_features = compute_env_features(centroids, gdf_env)

crimes_with_env_info = crimes_m.merge(env_centroid_features, on="centroid_id", how="left")

crimes_with_env_info.head()

,INCIDENT_REPORT_ID,LOCATION,ZIP,LATITUDE_PUBLIC,LONGITUDE_PUBLIC,CMPD_PATROL_DIVISION,NPA,LOCATION_TYPE_DESCRIPTION,PLACE_TYPE_DESCRIPTION,PLACE_DETAIL_DESCRIPTION,...,centroid_id,nearest_tree_dist_m,trees_within_10m,trees_within_25m,trees_within_50m,geometry,env_within_10m,env_within_25m,env_within_50m,env_object
0,20220704-0820-00,300 S BREVARD ST,28202,35.22,-80.84,Central,476,Outdoors,Open Area,Street/Highway,...,111,3.523966,3,21,51,POINT (514354.062 3897785.544),0,0,0,0
1,20220704-0820-00,300 S BREVARD ST,28202,35.22,-80.84,Central,476,Outdoors,Open Area,Street/Highway,...,111,3.523966,3,21,51,POINT (514354.062 3897785.544),0,0,0,0
2,20220704-0820-00,300 S BREVARD ST,28202,35.22,-80.84,Central,476,Outdoors,Open Area,Street/Highway,...,111,3.523966,3,21,51,POINT (514354.062 3897785.544),0,0,0,0
3,20230104-1325-00,400 S TRYON ST,28202,35.22,-80.85,Central,476,Outdoors,Commercial Place,Other - Commercial Place,...,92,15.400769,0,12,46,POINT (513979.527 3897997.11),0,4,10,10
4,20230104-1325-00,400 S TRYON ST,28202,35.22,-80.85,Central,476,Outdoors,Commercial Place,Other - Commercial Place,...,92,15.400769,0,12,46,POINT (513979.527 3897997.11),0,4,10,10


In [197]:
env_centroid_features = env_centroid_features.drop_duplicates(subset=["centroid_id"])

crime_gdf = crime_gdf.merge(
    env_centroid_features,
    on="centroid_id",
    how="left"
)

crime_gdf.head()

,INCIDENT_REPORT_ID,LOCATION,ZIP,LATITUDE_PUBLIC,LONGITUDE_PUBLIC,CMPD_PATROL_DIVISION,NPA,LOCATION_TYPE_DESCRIPTION,PLACE_TYPE_DESCRIPTION,PLACE_DETAIL_DESCRIPTION,...,centroid_id,nearest_tree_dist_m,trees_within_10m,trees_within_25m,trees_within_50m,geometry,env_within_10m,env_within_25m,env_within_50m,env_object
0,20220704-0820-00,300 S BREVARD ST,28202,35.22,-80.84,Central,476,Outdoors,Open Area,Street/Highway,...,111,3.523966,3,21,51,POINT (-80.84227 35.22301),0,0,0,0
1,20220704-0820-00,300 S BREVARD ST,28202,35.22,-80.84,Central,476,Outdoors,Open Area,Street/Highway,...,111,3.523966,3,21,51,POINT (-80.84227 35.22301),0,0,0,0
2,20220704-0820-00,300 S BREVARD ST,28202,35.22,-80.84,Central,476,Outdoors,Open Area,Street/Highway,...,111,3.523966,3,21,51,POINT (-80.84227 35.22301),0,0,0,0
3,20230104-1325-00,400 S TRYON ST,28202,35.22,-80.85,Central,476,Outdoors,Commercial Place,Other - Commercial Place,...,92,15.400769,0,12,46,POINT (-80.84638 35.22492),0,4,10,10
4,20230104-1325-00,400 S TRYON ST,28202,35.22,-80.85,Central,476,Outdoors,Commercial Place,Other - Commercial Place,...,92,15.400769,0,12,46,POINT (-80.84638 35.22492),0,4,10,10


In [207]:
gdf_unique = crime_gdf.drop_duplicates(subset="INCIDENT_REPORT_ID")
gdf_unique.head()

,INCIDENT_REPORT_ID,LOCATION,ZIP,LATITUDE_PUBLIC,LONGITUDE_PUBLIC,CMPD_PATROL_DIVISION,NPA,LOCATION_TYPE_DESCRIPTION,PLACE_TYPE_DESCRIPTION,PLACE_DETAIL_DESCRIPTION,...,centroid_id,nearest_tree_dist_m,trees_within_10m,trees_within_25m,trees_within_50m,geometry,env_within_10m,env_within_25m,env_within_50m,env_object
0,20220704-0820-00,300 S BREVARD ST,28202,35.22,-80.84,Central,476,Outdoors,Open Area,Street/Highway,...,111,3.523966,3,21,51,POINT (-80.84227 35.22301),0,0,0,0
3,20230104-1325-00,400 S TRYON ST,28202,35.22,-80.85,Central,476,Outdoors,Commercial Place,Other - Commercial Place,...,92,15.400769,0,12,46,POINT (-80.84638 35.22492),0,4,10,10
5,20180916-0854-01,900 SPINDLE ST,28206,35.23,-80.83,Central,476,Indoors,Residential,Private Residence,...,181,13.158830,0,4,5,POINT (-80.8316 35.23093),0,0,1,1
6,20180523-1548-04,600 S TRYON ST,28202,35.22,-80.85,Central,476,Outdoors,Open Area,Construction Site,...,81,30.097797,0,0,8,POINT (-80.84852 35.22346),0,1,9,9
7,20191125-1133-02,200 E BLAND ST,28203,35.22,-80.85,Central,3,Other,Commercial Place,Bar/Tavern/Nightclub,...,54,11.238243,0,3,7,POINT (-80.85429 35.21701),0,0,1,1


In [209]:
gdf_unique["INCIDENT_REPORT_ID"].value_counts()

INCIDENT_REPORT_ID
20220704-0820-00    1
20230104-1325-00    1
20180916-0854-01    1
20180523-1548-04    1
20191125-1133-02    1
                   ..
20231201-0237-01    1
20200131-1036-00    1
20221001-0930-00    1
20180628-1625-00    1
20190104-1026-01    1
Name: count, Length: 34753, dtype: int64

In [212]:
# Save as csv and geojson
gdf_unique.to_file("data/crimes_with_env_info.geojson", driver="GeoJSON")
gdf_unique.drop(columns="geometry").to_csv("data/crimes_with_env_info.csv", index=False)